In [ ]:
import os
import pandas as pd
import librosa
import soundfile as sf
import numpy as np
import random
import shutil
import logging
from sklearn.model_selection import train_test_split

# -----------------------------------------------------------------------------
# Configuration & Constants
# -----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

ROOT = r'D:\IMADS\CoffeeGrinder'
SUBDS = 'STWINonTable'
METADATA_FILE = os.path.join(ROOT, '2025 dataset planning.xlsx')
BASE_DIR = os.path.join(ROOT, SUBDS)
SEGMENTS_DIR = os.path.join(BASE_DIR, 'segments')
DURATION_IN_SECONDS = 5
SAMPLING_RATE = 16000
TARGET_SNR_DB = -7

# Mapping for renaming/dropping columns based on subset
COLUMN_MAP = {
    'STWINonGrinder': ('ID acq machine', 'ID acq table'),
    'STWINonTable': ('ID acq table', 'ID acq machine'),
}

# -----------------------------------------------------------------------------
# Domain Shift Parameters
# -----------------------------------------------------------------------------
SOURCE_DOMAIN_GRINDS = [0, 100, 200]
BACKGROUND_DOMAIN_MAPPING = {
    'A': 'source',
    'B': 'source',
    'C': 'source',
    'D': 'source',
    'E': 'target',
    'Z': 'target',
}

<h2>Step 1: Read and preprocess the metadata config.</h2>

In [ ]:
# -----------------------------------------------------------------------------
# Helper Functions - Excel 
# -----------------------------------------------------------------------------
def read_excel_sheet(filename, sheet_name):
    """Reads an Excel sheet and returns a DataFrame."""
    try:
        df = pd.read_excel(filename, sheet_name=sheet_name)
        return df
    except Exception as e:
        logging.error(f"Error reading sheet '{sheet_name}' from file '{filename}': {e}")
        raise

def process_config_dataframe(df, subset):
    """
    Process the config DataFrame:
    - Retrieve relevant columns,
    - Rename a key column to 'ID acq'
    """
    df = df.loc[:, ['Bkg noise', 'Type', 'Model', 'pos', 'grinder precision (um)','ID acq table', 'ID acq machine']]
    
    # Rename and drop columns depending on subset
    rename_col, drop_col = COLUMN_MAP.get(subset, (None, None))
    if drop_col and drop_col in df.columns:
        df = df.drop(columns=[drop_col])
    if rename_col and rename_col in df.columns:
        df.rename(columns={rename_col: 'ID acq'}, inplace=True)
    else:
        logging.warning(f"Column '{rename_col}' not found; skipping rename.")
            
    return df

In [ ]:
# Step 1: Read and preprocess the metadata config.
config_df = read_excel_sheet(METADATA_FILE, 'Config')
config_df = process_config_dataframe(config_df, SUBDS)

config_df

<h2>Step 2: Build the primary dataset and background DataFrames.</h2>

In [ ]:
# -----------------------------------------------------------------------------
# Helper Functions - Acquisition Dataframes 
# -----------------------------------------------------------------------------
def build_datasets(config_df, base_dir):
    """
    Constructs dataset and background DataFrames by iterating over the config_df.
    Uses each row to construct two file paths (for two microphones).
    """
    dataset_records = []
    background_records = []
    
    # Only process rows where 'ID acq' is non-null
    config_df = config_df[config_df['ID acq'].notna()]
    
    for _, row in config_df.iterrows():
        acq_id = row['ID acq']
        file_folder = os.path.join(base_dir, str(acq_id), '_Exported')
        mic1_file = os.path.join(file_folder, 'imp23absu_mic.wav')
        mic2_file = os.path.join(file_folder, 'imp34dt05_mic.wav')
        
        is_background = (not pd.isna(row.get('Bkg noise'))) and (str(row.get('Bkg noise')).strip() != '\\')
        
        if is_background:
            for mic_file in [mic1_file, mic2_file]:
                bkg_id = str(row.get('Bkg noise')).strip()
                domain_flag = BACKGROUND_DOMAIN_MAPPING.get(bkg_id, 'unknown')
                background_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'pos': row.get('pos'),
                    'background ID': bkg_id,
                    'domain_flag': domain_flag                    
                })
        else:
            for mic_file in [mic1_file, mic2_file]:
                domain_flag = 'source' if row.get('grinder precision (um)') in SOURCE_DOMAIN_GRINDS else 'target'
                dataset_records.append({
                    'ID acq': acq_id,
                    'file path': mic_file,
                    'Model': row.get('Model'),
                    'Type': row.get('Type'),
                    'position': row.get('pos'),
                    'grind size': row.get('grinder precision (um)'),
                    'source_target_flag': domain_flag  # domain flag for segments.
                })
    
    dataset_df = pd.DataFrame(dataset_records)
    background_df = pd.DataFrame(background_records)
    return dataset_df, background_df

In [ ]:
 # Step 2: Build the primary dataset and background DataFrames.
dataset_df, background_df = build_datasets(config_df, BASE_DIR)
dataset_df.to_csv(os.path.join(BASE_DIR, 'dataset.csv'), index=False)
background_df.to_csv(os.path.join(BASE_DIR, 'background.csv'), index=False)
logging.info("Datasets built and saved.")

dataset_df

<h2>Step 3: Export audio segments into files and build segments metadata.</h2>

In [ ]:
# -----------------------------------------------------------------------------
# Helper Functions - Extract Segments 
# -----------------------------------------------------------------------------
def extract_mic_type(file_path):
    """
    Extracts microphone type from a file path. Assumes the file name is of the form: imp23absu_mic.wav
    """
    basename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(basename)[0]
    return name_without_ext
 
def export_segments(df, segments_dir, duration_in_seconds, sampling_rate):
    """
    Reads each audio file from df, segments the file into fixed duration chunks,
    writes the segments to the output directory, and returns a DataFrame with metadata.
    """
    os.makedirs(segments_dir, exist_ok=True)
    records = []
   
    cnt = 0
    source_domain_grinds = [0, 100, 200]
 
    for _, row in df.iterrows():
        try:
            y, sr = librosa.load(row['file path'], sr=None)
        except Exception as e:
            logging.error(f"Error loading file {row['file path']}: {e}")
            continue

        if sr != sampling_rate:
            logging.warning(f"File {row['file path']} has sample rate {sr}, expected {sampling_rate}.")
        segment_length = duration_in_seconds * sr

        segments = [y[i:i + segment_length] for i in range(0, len(y), segment_length)
                    if len(y[i:i + segment_length]) == segment_length]
 
        normal_anomaly_flag = 'normal' if row.get('Type') == 'Normal' else 'anomaly'
        mic_type = extract_mic_type(row['file path'])
        domain_flag = row.get('source_target_flag')
 
        for segment in segments:
            filename = (
                f'section_00_{domain_flag}_train_{normal_anomaly_flag}_'
                f'{str(cnt).zfill(4)}_mic_{mic_type}_pos_{row.get("position")}_grind_{row.get("grind size")}.wav'
            )
            file_out_path = os.path.join(segments_dir, filename)
            try:
                sf.write(file_out_path, segment, sr)
            except Exception as e:
                logging.error(f"Error writing segment {file_out_path}: {e}")
                continue
            
            record = {
                'ID acq': row['ID acq'],
                'file path': os.path.join('segments', filename),  # relative path for later use
                'Model': row.get('Model'),
                'Type': row.get('Type'),
                'position': row.get('position'),
                'grind size': row.get('grind size'),
                'source_target_flag': domain_flag,
                'train_test_flag': 'train',
                'normal_anomaly_flag': normal_anomaly_flag,
                'mic_type': mic_type
            }
            records.append(record)
            cnt += 1
 
    segments_df = pd.DataFrame(records)
    return segments_df

In [ ]:
# Step 3: Export audio segments into files and build segments metadata.
segments_df = export_segments(dataset_df, SEGMENTS_DIR, DURATION_IN_SECONDS, SAMPLING_RATE)
segments_df.to_csv(os.path.join(BASE_DIR, 'segments.csv'), index=False)
logging.info("Audio segments exported and metadata saved.")

In [ ]:
# if segments df have already been created, just import segments_df
segments_df = pd.read_csv(os.path.join(BASE_DIR, 'segments.csv'))
segments_df

<h2>Step 4: Mix background noise into segments.</h2>

In [ ]:
# -----------------------------------------------------------------------------
# Helper Functions - Mix Background Noise
# -----------------------------------------------------------------------------

def mix_background_noise(segments_df, background_df, new_segments_dir, base_dir, duration_in_seconds, sampling_rate, target_snr_db):
    """
    Mixes background noise into each target segment.
    Uses caching to avoid recalculating durations and maintains a read head per background file.
    Updates the segments_df with the applied background 'background ID'.
    """
    bkg_duration_cache = {}
    bkg_read_heads = {}  # Structure: { mic_type: {acq_id: current_sample} }

    mic_types = background_df['file path'].apply(extract_mic_type).unique()
    for mic in mic_types:
        acq_ids = background_df['ID acq'].unique()
        bkg_read_heads[mic] = {acq: 0 for acq in acq_ids}

    segments_mixed_df = segments_df.copy()
    # get new segments directory from new_segments_dir
    new_segments_dir_name = os.path.basename(new_segments_dir)

    segments_df = segments_df.sample(frac=1, random_state=42).reset_index(drop=True)


    for i, row in segments_df.iterrows():
        segment_pos = row.get('position')
        segment_mic_type = row.get('mic_type')
        segment_domain = row.get('source_target_flag')
        
        # Filter background rows matching both position and mic type.
        bkg_candidates = background_df[
            (background_df['pos'] == segment_pos) &
            (background_df['file path'].str.contains(segment_mic_type)) &
            (background_df['domain_flag'] == segment_domain)
        ]
        if bkg_candidates.empty:
            logging.warning(f"No background candidates for segment index {i}.")
            continue

        legal_bkg_records = []
        for acq_id in bkg_candidates['ID acq'].unique():
            bkg_row = bkg_candidates[bkg_candidates['ID acq'] == acq_id].iloc[0] 
            bkg_file = bkg_row['file path']
            if bkg_file not in bkg_duration_cache:
                try:
                    duration_samples = librosa.get_duration(path=bkg_file) * sampling_rate
                    bkg_duration_cache[bkg_file] = duration_samples
                except Exception as e:
                    logging.error(f"Failed to get duration for {bkg_file}: {e}")
                    continue
            else:
                duration_samples = bkg_duration_cache[bkg_file]
            
            current_read_head = bkg_read_heads.get(segment_mic_type, {}).get(acq_id, 0)
            if current_read_head != -1 and (current_read_head + duration_in_seconds * sampling_rate) <= duration_samples:
                legal_bkg_records.append(bkg_row)
        
        if not legal_bkg_records:
            logging.warning(f"No legal background file available for segment index {i}.")
            continue

        legal_bkg_df = pd.DataFrame(legal_bkg_records)
        shuffled_legal = legal_bkg_df.sample(frac=1)  # randomize order (without fixed seed)
        bkg_row = shuffled_legal.iloc[0]
        bkg_acq_id = bkg_row['ID acq']
        bkg_file = bkg_row['file path']
        bkg_id = bkg_row['background ID']
        
        current_read_head = bkg_read_heads.get(segment_mic_type, {}).get(bkg_acq_id, 0)
        start_idx = current_read_head
        end_idx = start_idx + duration_in_seconds * sampling_rate
        
        try:
            bkg_audio, _ = librosa.load(bkg_file, sr=sampling_rate, offset=start_idx / sampling_rate, duration=duration_in_seconds)
        except Exception as e:
            logging.error(f"Error loading background audio chunk from {bkg_file}: {e}")
            continue
        
        # Update read head
        if end_idx >= bkg_duration_cache[bkg_file]:
            bkg_read_heads[segment_mic_type][bkg_acq_id] = -1
        else:
            bkg_read_heads[segment_mic_type][bkg_acq_id] = end_idx
        
        # Load target segment audio
        target_file_path = os.path.join(base_dir, row['file path'])
        try:
            target_audio, _ = librosa.load(target_file_path, sr=sampling_rate)
        except Exception as e:
            logging.error(f"Error loading target audio {target_file_path}: {e}")
            continue
        
        # Compute scaling factor using SNR
        signal_power = np.mean(target_audio ** 2)
        noise_power = np.mean(bkg_audio ** 2)
        target_noise_power = signal_power / (10 ** (target_snr_db / 10))
        scaling_factor = np.sqrt(target_noise_power / (noise_power + 1e-10))
        scaled_bkg = bkg_audio * scaling_factor
        mixed_audio = target_audio + scaled_bkg
        
        # Update the DataFrame with the applied background noise ID
        segments_mixed_df.at[i, 'background ID'] = bkg_id
        
        # Save mixed audio: update the file name to include the background ID.
        old_filepath = row['file path']
        mixed_filename = old_filepath.replace('.wav', f'_bkg_{bkg_id}.wav')
        mixed_filepath = os.path.join(new_segments_dir, os.path.basename(mixed_filename))
        try:
            sf.write(mixed_filepath, mixed_audio, sampling_rate)
        except Exception as e:
            logging.error(f"Error writing mixed audio to {mixed_filepath}: {e}")
            continue

        segments_mixed_df.at[i, 'file path'] = os.path.join(new_segments_dir_name, os.path.basename(mixed_filename))  # update relative path
            
    return segments_mixed_df

In [ ]:
# make new folder for segments with background noise
SEGMENTS_BKG_NAME = f'segments_bkg{TARGET_SNR_DB}dB'
segments_bkg_dir = os.path.join(BASE_DIR, SEGMENTS_BKG_NAME)

os.makedirs(segments_bkg_dir, exist_ok=True)

# Step 4: Mix background noise into segments.
segments_mixed_df = mix_background_noise(segments_df, background_df, segments_bkg_dir, BASE_DIR, DURATION_IN_SECONDS, SAMPLING_RATE, TARGET_SNR_DB)

<h2>Step 5: Rename anomaly files (change 'train' to 'test') and update metadata.</h2>

In [ ]:
def rename_anomaly_files(segments_df, base_dir):
    """
    For anomaly segments, renames the file path by replacing 'train' with 'test'
    and updates the DataFrame accordingly.
    """
    for i, row in segments_df.iterrows():
        if 'anomaly' in row.get('file path', ''):
            old_rel = row['file path']
            new_rel = old_rel.replace('train', 'test')
            old_abs = os.path.join(base_dir, old_rel)
            new_abs = os.path.join(base_dir, new_rel)
            try:
                os.rename(old_abs, new_abs)
            except Exception as e:
                logging.error(f"Error renaming {old_abs} to {new_abs}: {e}")
                continue
            segments_df.at[i, 'file path'] = new_rel
            segments_df.at[i, 'train_test_flag'] = 'test'
    return segments_df

In [ ]:
# Step 5: Rename anomaly files (change 'train' to 'test') and update metadata.
segments_mixed_df = rename_anomaly_files(segments_mixed_df, BASE_DIR)

<h2>Step 6: Create  new column for stratification labels.</h2>

In [ ]:
segments_mixed_df = pd.read_csv(os.path.join(BASE_DIR, 'segments_bkg-3dB.csv'))

In [ ]:
# Step 6: Convert DataFrame columns to string and build single-stage target label.
segments_mixed_df = segments_mixed_df.astype(str)
segments_mixed_df['target_label'] = (
    segments_mixed_df['source_target_flag'] + '_' +
    segments_mixed_df['grind size'] + '_' +
    segments_mixed_df['position'] + '_' +
    segments_mixed_df['mic_type'] + '_' +
    segments_mixed_df['background ID']
)

segments_mixed_df

In [ ]:
# filter occurrences of segments_mixed_df['target_label'].unique() which last letter is E

segments_mixed_df[segments_mixed_df['target_label'].str.contains('Z')]['target_label'].unique()

In [ ]:
# count the number of occurrences in target_label
target_label_counts = segments_mixed_df['target_label'].value_counts()
target_label_counts_df = target_label_counts.reset_index()
target_label_counts_df.columns = ['target_label', 'count']
target_label_counts_df

# visualize barplot of target_label counts
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(20, 6))
sns.barplot(data=target_label_counts_df, x='target_label', y='count')
plt.xticks(rotation=90)
plt.xlabel('Target Label')


<h2>Step 7: Split into train and test sets using stratification based on target_label.</h2>

In [ ]:
def stratified_sample(df, target_column, target_samples):
    """
    Performs stratified sampling on the DataFrame to achieve a total of target_samples overall.
    """
    class_counts = df[target_column].value_counts()
    class_ratios = class_counts / class_counts.sum()
    samples_per_class = (class_ratios * target_samples).round().astype(int)
    
    sampled_dfs = []
    for cls in class_counts.index:
        available = class_counts.loc[cls]
        n_samples = min(samples_per_class[cls], available)
        try:
            sampled = df[df[target_column] == cls].sample(n=n_samples, random_state=42)
            sampled_dfs.append(sampled)
        except Exception as e:
            logging.error(f"Error sampling class {cls}: {e}")
    return pd.concat(sampled_dfs) if sampled_dfs else pd.DataFrame()

def split_train_test(segments_df, test_size=30):
    """
    Splits the segments DataFrame into train and test sets.
    Stratification is based on a single-stage target_label that now includes
    background noise information.
    """

    segments_normal = segments_df[segments_df['normal_anomaly_flag'] == 'normal']
    segments_anomaly = segments_df[segments_df['normal_anomaly_flag'] == 'anomaly']
    print(f"Normal segments: {len(segments_normal)}, Anomaly segments: {len(segments_anomaly)}")
    
    normal_source = segments_normal[segments_normal['source_target_flag'] == 'source']
    normal_target = segments_normal[segments_normal['source_target_flag'] == 'target']
    print(f"Normal source segments: {len(normal_source)}, Normal target segments: {len(normal_target)}")
    
    if not normal_source.empty:
        train_ns, test_ns = train_test_split(
            normal_source,
            test_size=test_size,
            stratify=normal_source['target_label'],
            random_state=42
        )
    else:
        train_ns, test_ns = pd.DataFrame(), pd.DataFrame()
    
    if not normal_target.empty:
        train_nt, test_nt = train_test_split(
            normal_target,
            test_size=test_size,
            stratify=normal_target['target_label'],
            random_state=42
        )
    else:
        train_nt, test_nt = pd.DataFrame(), pd.DataFrame()
    
    test_anomaly = stratified_sample(segments_anomaly, 'target_label', target_samples=100)
    
    train_df = pd.concat([train_ns, train_nt], ignore_index=True)
    test_df = pd.concat([test_ns, test_nt, test_anomaly], ignore_index=True)
    
    return train_df, test_df

In [ ]:
# Step 7: Split into train and test sets using stratification based on target_label.
train_df, test_df = split_train_test(segments_mixed_df, test_size=108)
logging.info("Train/test splitting completed.")

# show the number of samples in each set
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")
# show the number of samples in each class
print("Train class distribution:")
print(train_df['target_label'].value_counts())
print("Test class distribution:")
print(test_df['target_label'].value_counts())


<h2>Step 8: Copy (and rename) files into train and test directories.</h2>

In [ ]:
def copy_files_to_dirs(train_df, test_df, base_dir, segments_dir):
    """
    Copies segment files into separate 'train' and 'test' directories.
    Files in the test set are renamed from 'train' to 'test' if needed.
    """
    train_dir = os.path.join(segments_dir, 'train')
    test_dir = os.path.join(segments_dir, 'test')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    for file_rel in train_df['file path']:
        src = os.path.join(base_dir, file_rel)
        dst = os.path.join(train_dir, os.path.basename(file_rel))
        try:
            shutil.copy(src, dst)
        except Exception as e:
            logging.error(f"Error copying {src} to {dst}: {e}")
    
    for i, row in test_df.iterrows():
        old_rel = row['file path']
        new_rel = old_rel.replace('train', 'test')
        old_abs = os.path.join(base_dir, old_rel)
        new_abs = os.path.join(base_dir, new_rel)
        try:
            os.rename(old_abs, new_abs)
        except Exception as e:
            logging.error(f"Error renaming {old_abs} to {new_abs}: {e}")
            continue
        test_df.at[i, 'file path'] = new_rel
        test_df.at[i, 'train_test_flag'] = 'test'
        try:
            shutil.copy(new_abs, test_dir)
        except Exception as e:
            logging.error(f"Error copying {new_abs} to {test_dir}: {e}")
    
    return train_df, test_df

In [ ]:
# Step 8: Copy (and rename) files into train and test directories.
train_df, test_df = copy_files_to_dirs(train_df, test_df, BASE_DIR, segments_bkg_dir)

# Step 9: Save the final train and test DataFrames to CSV files.
segments_mixed_df.to_csv(os.path.join(BASE_DIR, SEGMENTS_BKG_NAME) +'.csv', index=False)

test_count = segments_mixed_df['file path'].str.contains('test').sum()
logging.info(f"Number of test files in segments metadata: {test_count}")